In [1]:
import os
### fill in the token - DO NOT COMMIT! #########################
os.environ["HF_TOKEN"] = "hf_xxx"
################################################################

In [2]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA


path = "../data/output/responses.parquet"

pf = pq.ParquetFile(path)
print(pf.metadata)          # rows, row-groups

print('\n')

print(pf.schema_arrow)      # columns + types — verify your schema landed

/home/nfcom/Projects/Repositories/AIS/nla_inference_pipeline/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  created_by: parquet-cpp-arrow version 24.0.0
  num_columns: 14
  num_rows: 150
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 5732


prompt_id: string
condition: string
pair_id: int32
gen_idx: int32
prompt_text: string
response_text: string
prompt_len: int32
token_ids: list<element: int32>
  child 0, element: int32
token_strs: list<element: string>
  child 0, element: string
token_logprob: list<element: float>
  child 0, element: float
h_norm: list<element: float>
  child 0, element: float
model: string
layer: int32
gen_config: string


In [3]:
df = pq.read_table(path).to_pandas()

print(len(df), "rows")
print(df["condition"].value_counts(),'\n')        # sanity: A/B1/B2 counts

# one record, scalar fields only
print('random record inspect')
r = df.iloc[np.random.randint(0,100)]
print(r["prompt_id"], "|", r["condition"])
print("response:", r["response_text"][:200])
print("prompt_len:", r["prompt_len"], "| seq_len:", len(r["token_ids"]))
print("norm range:", min(r["h_norm"]), "-", max(r["h_norm"]))

150 rows
condition
A     50
B1    50
B2    50
Name: count, dtype: int64 

random record inspect
B1_05 | B1
response: Plants can exhibit behaviors that might be interpreted as suffering, although they don't have the nervous system or brain that animals do. Plants can respond to various stimuli and environmental chang
prompt_len: 33 | seq_len: 283
norm range: 83.55037 - 14333.698


In [4]:
## Inspect high norm tokens to make sure they are just structural artifacts
df['tok_norm_map'] = df.apply(
    lambda r: list(zip(r['token_strs'], r['h_norm'])), axis=1
)
df['max_norm_tokens'] = df['tok_norm_map'].apply(lambda x: [y for i,y in enumerate(x) if y[1]>14000])

## Response structure analysis

**Purpose:** validate the condition design (A / B1 / B2) before committing the token-selection protocol. Unsupervised structure checks whether the conditions are well-formed.

### 1. Responses separation by key dimensions

In [5]:
EMB_PATH = Path("../data/output/embeddings.npy")

if EMB_PATH.exists():
    E = np.load(EMB_PATH)
    print(f"loaded cached embeddings: {E.shape}")
else:
    print("no cached embeddings — encoding responses...")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    texts = df["response_text"].tolist()
    E = model.encode(texts, normalize_embeddings=True,
                     show_progress_bar=True, batch_size=32)
    np.save(EMB_PATH, E)
    print(f"encoded and saved: {E.shape}")

assert len(df) == E.shape[0], "df/E row mismatch"

loaded cached embeddings: (150, 384)


In [6]:
# PCA - 3D
pca = PCA(n_components=3)
coords = pca.fit_transform(E)
print("explained variance:", pca.explained_variance_ratio_)
print("cumulative:", pca.explained_variance_ratio_.cumsum())

df["pc1_3d"], df["pc2_3d"], df["pc3_3d"] = coords[:, 0], coords[:, 1], coords[:, 2]

explained variance: [0.14860345 0.09833749 0.06911089]
cumulative: [0.14860345 0.24694094 0.31605184]


In [10]:
fig = px.scatter_3d(
    df,
    x="pc1_3d",
    y="pc2_3d",
    z="pc3_3d",
    color="condition",
    template="plotly_dark",
    title='Prompt separation - PCA'
)
fig.update_layout(
    width=800,
    height=800,
)
fig.show()

In [12]:
# PCA - 2D
pca = PCA(n_components=2)
coords = pca.fit_transform(E)
print("explained variance:", pca.explained_variance_ratio_)
print("cumulative:", pca.explained_variance_ratio_.cumsum())

df["pc1_2d"], df["pc2_2d"] = coords[:, 0], coords[:, 1]

explained variance: [0.14860345 0.09833749]
cumulative: [0.14860345 0.24694094]


In [13]:
fig = px.scatter(
    df,
    x="pc1_2d",
    y="pc2_2d",
    color="condition",
    template="plotly_dark",
    title='Prompt separation - PCA'
)
fig.update_layout(
    width=800,
    height=800,
)
fig.show()